In [8]:
import os
import sys

import pandas as pd
from scipy.stats import chi2_contingency, kruskal, friedmanchisquare, wilcoxon

In [9]:
thresholds = ['0.1','0.5','0.8','A','I','N']
hardware = ["kyiv", "brisbane", "sherbrooke"]
mutant_types = ["equiv", "normal", "balanced"]
metrics = ['C', 'H', 'J', 'T', 'F', 'E']
metric_names=('Chisquare', 'Hellinger', 'Jensen-shannon', 'Trace', 'Fidelity', 'Expectation Values')
output_type = {'ae': 'Dominant', 'qpeexact': 'Dominant', 'vqe': 'Dominant', 'qft': 'Diverse', 'qftentangled': 'Diverse', 'wstate': 'Diverse'}


# Merge and save DFs for equiv, normal and balanced

In [7]:
output_folder = 'results'
os.makedirs(output_folder, exist_ok=True)

for mutant_type in mutant_types:
    dataframes = []
    for threshold in thresholds:
        for hw in hardware:
            csv_path = f'results/dataframes/{hw}_{mutant_type}_{threshold}.csv'
            df = pd.read_csv(csv_path)
            df['hardware'] = hw
            df['threshold'] = threshold
            dataframes.append(df)
    
    complete_df = pd.concat(dataframes, ignore_index=True)
    selected_columns = complete_df[['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Input', 'Input_type', 'Algorithm', 'Qubits_number', 'Operator', 'Gate', 'Position', 'Qubits', 'Gate_type', 'Relative_position', 'Output_type', 'hardware', 'threshold']]
        
    new_rows = []

    for metric in metrics:
        # Extract true and predicted labels for the current metric
        true_labels = complete_df[f'Killed_I{metric}']
        predicted_labels = complete_df[f'Killed_N{metric}']
        metric_df = selected_columns.copy()
        metric_df['metric'] = metric  
        metric_df['expected'] = true_labels  
        metric_df['predicted'] = predicted_labels  
        metric_df['correctness'] = (true_labels == predicted_labels)  
        new_rows.append(metric_df)
    
    metric_df = pd.concat(new_rows, ignore_index=True)
    metric_df.reset_index(drop=True, inplace=True)

    output_path = os.path.join(output_folder, f'results_{mutant_type}_complete.csv')
    metric_df.to_csv(output_path, index=False)   


# Statistical Analysis

In [10]:
csv_path = f'results/results_balanced_selected.csv'
df_balanced = pd.read_csv(csv_path, dtype={'threshold': str})

csv_path = f'results/results_equiv_selected.csv'
df_equiv = pd.read_csv(csv_path, dtype={'threshold': str})

csv_path = f'results/results_normal_selected.csv'
df_normal = pd.read_csv(csv_path, dtype={'threshold': str})

df = pd.concat([df_equiv, df_normal], ignore_index=True)

In [11]:
df['threshold_metric'] = list(zip(df['threshold'], df['metric']))

# Separate categorical and numerical columns
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

for cat in ['threshold', 'metric', 'threshold_metric']:
    categorical_columns.remove(cat)  

print(categorical_columns)
print(numerical_columns)

['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Qubits_number', 'Position', 'Qubits']


In [12]:
def ratio_correct_counts(df, col):
    if 'threshold_metric' not in df.columns or 'correctness' not in df.columns:
        raise ValueError("The DataFrame must contain 'threshold_metric' and 'correctness' columns.")
    
    # Step 1: Group by number of qubits and threshold metric, then count the correct values
    total_counts = (
        df.groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='total_count')  # Total count for each group
    )
    
    correct_counts = (
        df[df['correctness'] == True]  # Filter only correct rows
        .groupby([col, 'threshold_metric'])
        .size()
        .reset_index(name='correct_count')  # Reset index and name the count column
    )
    
    correct_counts = pd.merge(correct_counts, total_counts, on=[col, 'threshold_metric'], how='left')
    
    # Step 4: Calculate the ratio of correct_count to total_count
    correct_counts['correct_ratio'] = correct_counts['correct_count'] / correct_counts['total_count']
    
    return correct_counts

### Friedman Chi Square test

In [13]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in categorical_columns: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        stat, p_value = friedmanchisquare(*grouped_ranks)
        print(f"P-value from Friedman test: {p_value}")
        print(f"Stat from Friedman test: {stat}")
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

For category Input:
P-value from Friedman test: 0.0
Stat from Friedman test: 2269.671811223121
Result: Significant association between 'cat' and 'threshold_metric'.
For category Input_type:
P-value from Wilcoxon test: 0.0011994246451649815
Stat from Wilcoxon test: 133.0
Result: Significant association between 'cat' and 'threshold_metric'.
For category Algorithm:
P-value from Friedman test: 8.976163404382791e-08
Stat from Friedman test: 41.095238095238074
Result: Significant association between 'cat' and 'threshold_metric'.
For category Operator:
P-value from Friedman test: 1.0613796652881946e-08
Stat from Friedman test: 36.72222222222217
Result: Significant association between 'cat' and 'threshold_metric'.
For category Gate:
P-value from Friedman test: 1.1064523877793631e-73
Stat from Friedman test: 409.2928184600896
Result: Significant association between 'cat' and 'threshold_metric'.
For category Gate_type:
P-value from Wilcoxon test: 0.01392684379243292
Stat from Wilcoxon test: 178.

In [14]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']

for cat in numerical_columns:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    l = len(correct_counts.groupby(cat))
    if l > 2:
        
        try:
            stat, p_value = friedmanchisquare(*grouped_ranks)
            print(f"P-value from Friedman test: {p_value}")
            print(f"Stat from Friedman test: {stat}")
        except Exception as e:
            print(f'Error processing column {cat}: {str(e)}')
        
    else:
        stat, p_value = wilcoxon(*grouped_ranks)
        print(f"P-value from Wilcoxon test: {p_value}")
        print(f"Stat from Wilcoxon test: {stat}")
        
    if p_value < 0.05:
        print("Result: Significant association between 'cat' and 'threshold_metric'.")
    else:
        print("Result: No significant association between 'cat' and 'threshold_metric'.")
    print(f"================================================================")
    

For category gates:
P-value from Friedman test: 3.303734748341364e-21
Stat from Friedman test: 170.54081128568194
Result: Significant association between 'cat' and 'threshold_metric'.
For category depth:
P-value from Friedman test: 6.697760811086974e-14
Stat from Friedman test: 107.07088205572387
Result: Significant association between 'cat' and 'threshold_metric'.
For category singlequbit_gates:
P-value from Friedman test: 1.3155588228365087e-08
Stat from Friedman test: 71.24422271919242
Result: Significant association between 'cat' and 'threshold_metric'.
For category multiqubit_gates:
P-value from Friedman test: 6.304474602303229e-15
Stat from Friedman test: 118.99579100145131
Result: Significant association between 'cat' and 'threshold_metric'.
For category Qubits_number:
P-value from Friedman test: 0.0034473175341959723
Stat from Friedman test: 19.464285714285666
Result: Significant association between 'cat' and 'threshold_metric'.
For category Position:
Error processing column Po

In [ ]:
x = [72, 96, 88, 92, 74, 76, 82]
y = [120, 120, 132, 120, 101, 96, 112]
z = [76, 95, 104, 96, 84, 72, 76]
res = friedmanchisquare(x, y, z)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(x, y, z)
print("Kruskal")
print(res.pvalue)
print(res.statistic)
print('')


x = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
y = [1, 20, 30, 40, 50, 60, 70, 80, 90, 100]
z = [5, 25, 35, 45, 55, 65, 75, 85, 95, 105]

array = [x, y, z]
transposed_array = [[row[i] for row in array] for i in range(len(x))]

res = friedmanchisquare(*array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*transposed_array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

print('')
res = friedmanchisquare(*transposed_array)
print("Friedman")
print(res.pvalue)
print(res.statistic)
res = kruskal(*array)
print("Kruskal")
print(res.pvalue)
print(res.statistic)

### Kruskal-Wallis test

In [ ]:
all_cats = numerical_columns  + categorical_columns

#['Input', 'Input_type', 'Algorithm', 'Operator', 'Gate', 'Gate_type', 'Relative_position', 'Output_type', 'hardware']
#['gates', 'depth', 'singlequbit_gates', 'multiqubit_gates', 'Position', 'Qubits']
significant = []
non_significant = []

for cat in all_cats: #['Gate_type']: # ['hardware', 'Operator', 'Algorithm', 'Gate']:   
    print(f"For category {cat}:")

    # Ranked DataFrame from previous steps
    correct_counts = ratio_correct_counts(df, cat)
    grouped_ranks = correct_counts.groupby(cat)['correct_ratio'].apply(list)
    
    # Perform Kruskal-Wallis test
    stat, p_value = kruskal(*grouped_ranks)
    
    print(f"P-value from Kruskal-Wallis test: {p_value}")
    print(f"Stat from Kruskal-Wallis test: {stat}")
        
    if p_value < 0.05:
        print(f"Result: Significant association between {cat} and 'threshold_metric'.")
        significant.append(cat)
    else:
        print(f"Result: No significant association between {cat} and 'threshold_metric'.")
        non_significant.append(cat)
    print(f"================================================================")
    
    
print(f"Significant association between the threshold_metric selection and {significant}.")
print(f"Non-Significant association between the threshold_metric and {non_significant}.")
    